A few important decisions:
* We'll retain business keys alongside surrogate keys.
* We'll use integer surrogate keys where appropriate.
* We'll avoid unnecessary columns and transformations.
* We'll document the grain of each dimension.
* We'll design SCD handling deliberately rather than introduce artificial complexity.
* dim_date will support all relevant order lifecycle dates.

In [0]:
tables = [
    "customers",
    "products",
    "sellers",
    "product_category_name_translation"
]

for table in tables:
    print(f"\n{'=' * 60}")
    print(f"TABLE: {table}")
    print("=" * 60)

    df = spark.table(f"lakehouse.silver.{table}")

    df.printSchema()
    display(df.limit(5))


In [0]:

from pyspark.sql import functions as F

# Load Silver tables
customers = spark.table("lakehouse.silver.customers")
products = spark.table("lakehouse.silver.products")
sellers = spark.table("lakehouse.silver.sellers")
category_translation = spark.table(
    "lakehouse.silver.product_category_name_translation"
)

# 1. Check business key uniqueness
tables = {
    "customers": (customers, "customer_id"),
    "products": (products, "product_id"),
    "sellers": (sellers, "seller_id"),
    "category_translation": (
        category_translation,
        "product_category_name"
    )
}

for name, (df, key) in tables.items():
    total = df.count()
    distinct_keys = df.select(key).distinct().count()
    null_keys = df.filter(F.col(key).isNull()).count()

    print(f"\n{name}")
    print(f"Total rows: {total}")
    print(f"Distinct keys: {distinct_keys}")
    print(f"Null keys: {null_keys}")
    print(f"Duplicate keys: {total - distinct_keys - null_keys}")


In [0]:
# 2. Check category translation coverage

product_categories = (
    products
    .select("product_category_name")
    .distinct()
)

translated_categories = (
    category_translation
    .select("product_category_name")
    .distinct()
)

untranslated = product_categories.join(
    translated_categories,
    on="product_category_name",
    how="left_anti"
)

print("Product categories without translation:")
display(untranslated)

print("Translation table:")
display(category_translation.orderBy("product_category_name"))


In [0]:
from pyspark.sql import functions as F

customers = spark.table("lakehouse.silver.customers")
products = spark.table("lakehouse.silver.products")

# Check whether one customer_id maps to multiple customer_unique_id values
customer_mapping_issues = (
    customers
    .groupBy("customer_id")
    .agg(
        F.countDistinct("customer_unique_id").alias("unique_id_count")
    )
    .filter(F.col("unique_id_count") > 1)
)

print(
    "Customer IDs mapping to multiple unique IDs:",
    customer_mapping_issues.count()
)

# Check how many products have a missing category
print(
    "Products with missing category:",
    products.filter(
        F.col("product_category_name").isNull()
    ).count()
)

# Check translation coverage
translations = spark.table(
    "lakehouse.silver.product_category_name_translation"
)

translation_coverage = (
    products
    .filter(F.col("product_category_name").isNotNull())
    .select("product_category_name")
    .distinct()
    .join(
        translations.select("product_category_name"),
        on="product_category_name",
        how="left_anti"
    )
)

print(
    "Distinct non-null categories without translation:",
    translation_coverage.count()
)

In [0]:
from pyspark.sql import functions as F

customers = spark.table("lakehouse.silver.customers")

customer_profile_variation = (
    customers
    .groupBy("customer_unique_id")
    .agg(
        F.count("*").alias("customer_record_count"),

        F.countDistinct(
            "customer_zip_code_prefix"
        ).alias("distinct_zip_codes"),

        F.countDistinct(
            "customer_city"
        ).alias("distinct_cities"),

        F.countDistinct(
            "customer_state"
        ).alias("distinct_states"),

        F.collect_set(
            "customer_zip_code_prefix"
        ).alias("zip_codes"),

        F.collect_set(
            "customer_city"
        ).alias("cities"),

        F.collect_set(
            "customer_state"
        ).alias("states")
    )
    .filter(
        (F.col("distinct_zip_codes") > 1) |
        (F.col("distinct_cities") > 1) |
        (F.col("distinct_states") > 1)
    )
)

print(
    "Customers with at least one varying attribute:",
    customer_profile_variation.count()
)

display(
    customer_profile_variation
    .orderBy(F.desc("customer_record_count"))
)

In [0]:
from pyspark.sql import functions as F

variation_summary = (
    customer_profile_variation
    .agg(
        F.count("*").alias("customers_with_variation"),

        F.sum(
            F.when(F.col("distinct_zip_codes") > 1, 1).otherwise(0)
        ).alias("different_zip_codes"),

        F.sum(
            F.when(F.col("distinct_cities") > 1, 1).otherwise(0)
        ).alias("different_cities"),

        F.sum(
            F.when(F.col("distinct_states") > 1, 1).otherwise(0)
        ).alias("different_states"),

        F.sum(
            F.when(
                (F.col("distinct_cities") > 1) |
                (F.col("distinct_states") > 1),
                1
            ).otherwise(0)
        ).alias("different_city_or_state")
    )
)

display(variation_summary)

In [0]:
display(
    customer_profile_variation
    .filter(
        (F.col("distinct_cities") > 1) |
        (F.col("distinct_states") > 1)
    )
    .orderBy(F.desc("customer_record_count"))
)

In [0]:
customer_order_history = (
    customers.alias("c")
    .join(
        spark.table("lakehouse.silver.orders").alias("o"),
        F.col("c.customer_id") == F.col("o.customer_id"),
        "inner"
    )
    .select(
        F.col("c.customer_unique_id"),
        F.col("c.customer_id"),
        F.col("c.customer_zip_code_prefix"),
        F.col("c.customer_city"),
        F.col("c.customer_state"),
        F.col("o.order_purchase_timestamp")
    )
)

display(
    customer_order_history
    .filter(
        F.col("customer_unique_id") ==
        "47c1a3033b8b77b3ab6e109eb4d5fdf3"
    )
    .orderBy("order_purchase_timestamp")
)

In [0]:
from pyspark.sql import functions as F

customer_geography = (
    customers
    .groupBy(
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    )
    .count()
)

print("Total distinct geography combinations:",
      customer_geography.count())

print("Combinations with missing attributes:",
      customer_geography.filter(
          F.col("customer_zip_code_prefix").isNull() |
          F.col("customer_city").isNull() |
          F.col("customer_state").isNull()
      ).count())

print("Distinct customer ZIP prefixes:",
      customers.select("customer_zip_code_prefix").distinct().count())

print("Distinct customer cities:",
      customers.select("customer_city").distinct().count())

print("Distinct customer states:",
      customers.select("customer_state").distinct().count())

In [0]:
geolocation = spark.table("lakehouse.silver.geolocation")

print("Total rows:", geolocation.count())

print(
    "Distinct ZIP prefixes:",
    geolocation.select("geolocation_zip_code_prefix").distinct().count()
)

print(
    "Distinct ZIP/city/state combinations:",
    geolocation.select(
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    ).distinct().count()
)

display(
    geolocation
    .select(
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    )
    .limit(10)
)

In [0]:
geolocation_zip_counts = (
    geolocation
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct(
            F.struct(
                "geolocation_lat",
                "geolocation_lng"
            )
        ).alias("distinct_coordinates")
    )
)

display(
    geolocation_zip_counts
    .filter(F.col("distinct_coordinates") > 1)
    .orderBy(F.desc("distinct_coordinates"))
)

In [0]:
customer_geography = (
    customers
    .select(
        F.col("customer_zip_code_prefix").alias("zip_code_prefix"),
        F.col("customer_city").alias("city"),
        F.col("customer_state").alias("state")
    )
    .distinct()
)

geo_combinations = (
    geolocation
    .select(
        F.col("geolocation_zip_code_prefix").alias("zip_code_prefix"),
        F.col("geolocation_city").alias("city"),
        F.col("geolocation_state").alias("state")
    )
    .distinct()
)

customer_without_geo = customer_geography.join(
    geo_combinations,
    on=["zip_code_prefix", "city", "state"],
    how="left_anti"
)

geo_without_customer = geo_combinations.join(
    customer_geography,
    on=["zip_code_prefix", "city", "state"],
    how="left_anti"
)

print("Customer combinations:", customer_geography.count())
print("Geolocation combinations:", geo_combinations.count())
print("Customer combinations without geolocation:", customer_without_geo.count())
print("Geolocation combinations without customer:", geo_without_customer.count())

display(customer_without_geo.limit(20))

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    desc
)

customer_without_geo.groupBy("state") \
    .agg(
        count("*").alias("unmatched_combinations"),
        countDistinct("zip_code_prefix").alias("unmatched_zip_prefixes"),
        countDistinct("city").alias("unmatched_cities")
    ) \
    .orderBy(desc("unmatched_combinations")) \
    .show(30, truncate=False)

In [0]:
unmatched_zip_analysis = (
    customer_without_geo.alias("c")
    .join(
        geolocation.select(
            col("geolocation_zip_code_prefix").alias("geo_zip")
        ).distinct().alias("g"),
        col("c.zip_code_prefix") == col("g.geo_zip"),
        "left"
    )
    .groupBy("c.state")
    .agg(
        count("*").alias("unmatched_combinations"),
        count("g.geo_zip").alias("zip_prefixes_found_in_geolocation")
    )
)

unmatched_zip_analysis.show(truncate=False)

In [0]:
seller_geography = (
    sellers.select(
        col("seller_zip_code_prefix").alias("zip_code_prefix"),
        col("seller_city").alias("city"),
        col("seller_state").alias("state")
    )
    .distinct()
)

print(
    "Distinct seller geography combinations:",
    seller_geography.count()
)

print(
    "Distinct seller ZIP prefixes:",
    seller_geography.select("zip_code_prefix").distinct().count()
)

In [0]:
seller_without_geo = seller_geography.join(
    geo_combinations,
    on=["zip_code_prefix", "city", "state"],
    how="left_anti"
)

print(
    "Seller combinations without geolocation:",
    seller_without_geo.count()
)

seller_without_geo.show(30, truncate=False)

In [0]:
seller_zip_coverage = (
    seller_without_geo.alias("s")
    .join(
        geolocation.select(
            col("geolocation_zip_code_prefix").alias("geo_zip")
        ).distinct().alias("g"),
        col("s.zip_code_prefix") == col("g.geo_zip"),
        "left"
    )
)

seller_zip_coverage.select(
    col("s.zip_code_prefix"),
    col("s.city"),
    col("s.state"),
    col("g.geo_zip")
).show(30, truncate=False)

# DIMENSIONS

In [0]:
from pyspark.sql import functions as F

customer_geography = (
    customers.select(
        F.col("customer_zip_code_prefix").alias("zip_code_prefix"),
        F.col("customer_city").alias("city"),
        F.col("customer_state").alias("state")
    )
)

seller_geography = (
    sellers.select(
        F.col("seller_zip_code_prefix").alias("zip_code_prefix"),
        F.col("seller_city").alias("city"),
        F.col("seller_state").alias("state")
    )
)

dim_geography = (
    customer_geography
    .unionByName(seller_geography)
    .distinct()
    .withColumn(
        "geography_sk",
        F.sha2(
            F.to_json(
                F.struct(
                    F.col("zip_code_prefix"),
                    F.col("city"),
                    F.col("state")
                )
            ),
            256
        )
    )
    .select(
        "geography_sk",
        "zip_code_prefix",
        "city",
        "state"
    )
)

In [0]:
print("Total rows:", dim_geography.count())

print(
    "Distinct geography keys:",
    dim_geography.select("geography_sk").distinct().count()
)

print(
    "Null geography keys:",
    dim_geography.filter(
        F.col("geography_sk").isNull()
    ).count()
)

print(
    "Duplicate geography keys:",
    dim_geography.groupBy("geography_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Rows with missing attributes:",
    dim_geography.filter(
        F.col("zip_code_prefix").isNull()
        | F.col("city").isNull()
        | F.col("state").isNull()
    ).count()
)

In [0]:
dim_geography.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse.gold.dim_geography")

In [0]:
dim_customers = (
    customers
    .select("customer_unique_id")
    .distinct()
    .withColumn(
        "customer_sk",
        F.sha2(F.col("customer_unique_id"), 256)
    )
    .select(
        "customer_sk",
        "customer_unique_id"
    )
)

In [0]:
# 1. Total rows
total_rows = dim_customers.count()

# 2. Distinct customer identifiers
distinct_customers = (
    dim_customers
    .select("customer_unique_id")
    .distinct()
    .count()
)

# 3. Null customer identifiers
null_customer_ids = (
    dim_customers
    .filter(F.col("customer_unique_id").isNull())
    .count()
)

# 4. Null surrogate keys
null_surrogate_keys = (
    dim_customers
    .filter(F.col("customer_sk").isNull())
    .count()
)

# 5. Duplicate surrogate keys
duplicate_surrogate_keys = (
    dim_customers
    .groupBy("customer_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total rows:", total_rows)
print("Distinct customer identifiers:", distinct_customers)
print("Null customer identifiers:", null_customer_ids)
print("Null surrogate keys:", null_surrogate_keys)
print("Duplicate surrogate keys:", duplicate_surrogate_keys)

In [0]:
incorrect_hashes = (
    dim_customers
    .filter(
        F.col("customer_sk") !=
        F.sha2(F.col("customer_unique_id"), 256)
    )
    .count()
)

print("Incorrect surrogate keys:", incorrect_hashes)

In [0]:
dim_customers.toPandas()

In [0]:
dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse.gold.dim_customers")

In [0]:
from pyspark.sql import functions as F

products = spark.table("lakehouse.silver.products")

category_translation = spark.table(
    "lakehouse.silver.product_category_name_translation"
)

dim_products = (
    products.alias("p")
    .join(
        category_translation.alias("t"),
        F.col("p.product_category_name")
        == F.col("t.product_category_name"),
        "left"
    )
    .select(
        F.sha2(F.col("p.product_id"), 256).alias("product_sk"),
        F.col("p.product_id"),
        F.col("p.product_category_name"),
        F.col("t.product_category_name_english"),
        F.col("p.product_name_lenght").alias("product_name_length"),
        F.col("p.product_description_lenght").alias("product_description_length"),
        F.col("p.product_photos_qty"),
        F.col("p.product_weight_g"),
        F.col("p.product_length_cm"),
        F.col("p.product_height_cm"),
        F.col("p.product_width_cm")
    )
)

In [0]:
# 1. Total rows
total_rows = dim_products.count()

# 2. Distinct product IDs
distinct_product_ids = (
    dim_products.select("product_id").distinct().count()
)

# 3. Null product IDs
null_product_ids = (
    dim_products.filter(F.col("product_id").isNull()).count()
)

# 4. Null surrogate keys
null_product_keys = (
    dim_products.filter(F.col("product_sk").isNull()).count()
)

# 5. Duplicate surrogate keys
duplicate_product_keys = (
    dim_products.groupBy("product_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 6. Incorrect surrogate keys
incorrect_hashes = (
    dim_products.filter(
        F.col("product_sk") != F.sha2(F.col("product_id"), 256)
    ).count()
)

# 7. Products without an English category translation
untranslated_categories = (
    dim_products
    .filter(
        F.col("product_category_name").isNotNull()
        & F.col("product_category_name_english").isNull()
    )
    .select("product_category_name")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Distinct product IDs:", distinct_product_ids)
print("Null product IDs:", null_product_ids)
print("Null surrogate keys:", null_product_keys)
print("Duplicate surrogate keys:", duplicate_product_keys)
print("Incorrect surrogate keys:", incorrect_hashes)
print("Untranslated non-null categories:", untranslated_categories)

In [0]:
dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse.gold.dim_products")

In [0]:
from pyspark.sql import functions as F

sellers = spark.table("lakehouse.silver.sellers")

dim_geography = spark.table("lakehouse.gold.dim_geography")

dim_sellers = (
    sellers.alias("s")
    .join(
        dim_geography.alias("g"),
        (
            F.col("s.seller_zip_code_prefix")
            == F.col("g.zip_code_prefix")
        )
        & (
            F.col("s.seller_city")
            == F.col("g.city")
        )
        & (
            F.col("s.seller_state")
            == F.col("g.state")
        ),
        "left"
    )
    .select(
        F.sha2(F.col("s.seller_id"), 256).alias("seller_sk"),
        F.col("s.seller_id"),
        F.col("g.geography_sk")
    )
)

In [0]:
# 1. Total rows
total_rows = dim_sellers.count()

# 2. Distinct seller IDs
distinct_seller_ids = (
    dim_sellers.select("seller_id").distinct().count()
)

# 3. Null seller IDs
null_seller_ids = (
    dim_sellers.filter(F.col("seller_id").isNull()).count()
)

# 4. Null surrogate keys
null_seller_keys = (
    dim_sellers.filter(F.col("seller_sk").isNull()).count()
)

# 5. Duplicate surrogate keys
duplicate_seller_keys = (
    dim_sellers.groupBy("seller_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 6. Incorrect hashes
incorrect_hashes = (
    dim_sellers.filter(
        F.col("seller_sk") != F.sha2(F.col("seller_id"), 256)
    ).count()
)

# 7. Missing geography references
missing_geography = (
    dim_sellers.filter(F.col("geography_sk").isNull()).count()
)

print("Total rows:", total_rows)
print("Distinct seller IDs:", distinct_seller_ids)
print("Null seller IDs:", null_seller_ids)
print("Null surrogate keys:", null_seller_keys)
print("Duplicate surrogate keys:", duplicate_seller_keys)
print("Incorrect hashes:", incorrect_hashes)
print("Missing geography references:", missing_geography)

In [0]:
dim_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse.gold.dim_sellers")

In [0]:
from pyspark.sql import functions as F

orders = spark.table("lakehouse.silver.orders")

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

In [0]:
date_values = None

for column_name in date_columns:
    current_dates = (
        orders
        .select(
            F.to_date(F.col(column_name)).alias("full_date")
        )
        .filter(F.col("full_date").isNotNull())
    )

    if date_values is None:
        date_values = current_dates
    else:
        date_values = date_values.unionByName(current_dates)

date_bounds = date_values.agg(
    F.min("full_date").alias("min_date"),
    F.max("full_date").alias("max_date")
).first()

min_date = date_bounds["min_date"]
max_date = date_bounds["max_date"]

print("Minimum date:", min_date)
print("Maximum date:", max_date)

In [0]:
date_range = spark.range(1).select(
    F.explode(
        F.sequence(
            F.lit(min_date).cast("date"),
            F.lit(max_date).cast("date"),
            F.expr("INTERVAL 1 DAY")
        )
    ).alias("full_date")
)

dim_date = (
    date_range
    .select(
        F.date_format("full_date", "yyyyMMdd")
            .cast("int").alias("date_sk"),

        F.col("full_date").alias("date"),

        F.year("full_date").alias("year"),

        F.quarter("full_date").alias("quarter"),

        F.month("full_date").alias("month"),

        F.date_format("full_date", "MMMM")
            .alias("month_name"),

        F.dayofmonth("full_date").alias("day_of_month"),

        F.dayofweek("full_date").alias("day_of_week"),

        F.date_format("full_date", "EEEE")
            .alias("day_name"),

        F.weekofyear("full_date").alias("week_of_year"),

        F.dayofweek("full_date")
            .isin(1, 7).alias("is_weekend"),

    )
)

In [0]:
total_rows = dim_date.count()

distinct_dates = (
    dim_date.select("date").distinct().count()
)

distinct_date_keys = (
    dim_date.select("date_sk").distinct().count()
)

null_dates = (
    dim_date.filter(F.col("full_date").isNull()).count()
)

null_date_keys = (
    dim_date.filter(F.col("date_sk").isNull()).count()
)

duplicate_date_keys = (
    dim_date.groupBy("date_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total rows:", total_rows)
print("Distinct dates:", distinct_dates)
print("Distinct date keys:", distinct_date_keys)
print("Null dates:", null_dates)
print("Null date keys:", null_date_keys)
print("Duplicate date keys:", duplicate_date_keys)

print("Expected calendar days:", (max_date - min_date).days + 1)

In [0]:
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse.gold.dim_date")

In [0]:
dim_date.toPandas()